# MotionBERT — Dance Genre Classification on AIST++
## ICCV 2023 — Transfer Learning for Dance

**MotionBERT** is a unified pretraining framework (Zhu et al., ICCV 2023) that learns human motion representations from 2D skeleton sequences.

**Key idea:** Pre-trained on massive motion datasets → fine-tune on AIST++ with just one linear layer.

**Architecture:** DSTformer (Dual-Stream Spatio-Temporal Transformer)
- Stream 1: Spatial attention (between joints in same frame)
- Stream 2: Temporal attention (across frames for same joint)

**Input:** 2D skeleton sequences (17 keypoints × T frames) → same as our existing data!

**Comparison with our previous models:**

| Model | Accuracy | Architecture |
|---|---|---|
| SVM | 85.96% | Traditional ML |
| RF | 85.96% | Traditional ML |
| LSTM | 91.23% | Recurrent |
| ST-GCN | 98.68% | Graph Conv |
| **MotionBERT** | **???** | **Pretrained Transformer** |

In [ ]:
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA:', torch.version.cuda)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Python: {__import__("sys").version}')


In [ ]:
!git clone https://github.com/Walter0807/MotionBERT.git --quiet
%cd MotionBERT
 
!pip install -r requirements.txt --quiet
 
print('MotionBERT installed')
!ls


In [ ]:
!pip install huggingface_hub --quiet
from huggingface_hub import hf_hub_download
import os
 
os.makedirs('checkpoint/pretrain', exist_ok=True)
 
try:
    path = hf_hub_download(
        repo_id='wentao-zhu/MotionBERT',
        filename='MB_release.bin',
        local_dir='checkpoint/pretrain'
    )
    print(f'Downloaded: {path}')
except:
    print('Trying alternative download...')
    !wget -q "https://dl.fbaipublicfiles.com/pytorch3d/data/MotionBERT/MB_release.bin" \
         -O checkpoint/pretrain/MB_release.bin
 
print('Pretrained weights ready')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
 
import sys
sys.path.insert(0, '/content/MotionBERT')
 
import os, pickle
import numpy as np
import pandas as pd
import time
 
PKL_DIR = '/content/drive/MyDrive/AIST_PKL'
AR_DIR  = '/content/drive/MyDrive/AIST_ActionRec_MotionBERT'
os.makedirs(AR_DIR, exist_ok=True)
 
GENRE_CODES = {
    'gBR': 'Break',    'gPO': 'Pop',        'gLO': 'Lock',
    'gWA': 'Waack',    'gMH': 'Mid Hip-hop', 'gLH': 'LA Hip-hop',
    'gHO': 'House',    'gKR': 'Krump',       'gJS': 'Street Jazz',
    'gJB': 'Ballet Jazz'
}
GENRES       = list(GENRE_CODES.values())
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
NUM_CLASSES  = len(GENRES)
 
all_files = sorted(os.listdir(PKL_DIR))
print(f'Total .pkl files: {len(all_files)}')


In [ ]:
 
WINDOW_SIZE = 243
STRIDE      = 81
 
 
def normalize_skeleton(keypoints):
    """Normalize by hip center, scale to [-1,1]."""
    xy = keypoints[:, :2].copy()
    if np.all(xy == 0) or np.any(np.isnan(xy)):
        return np.zeros_like(keypoints)
    hip_center = (xy[11] + xy[12]) / 2.0
    xy -= hip_center
    max_val = np.abs(xy).max()
    if max_val > 0:
        xy /= max_val
    result = keypoints.copy()
    result[:, :2] = xy
    return result
 
 
def load_sequence(pkl_path, camera_idx=0, frame_step=3):
    """Load AIST++ sequence. frame_step=3: 60fps→20fps (MotionBERT prefers higher fps)."""
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    kpts = data['keypoints2d'][camera_idx]
    return [normalize_skeleton(kpts[i]) for i in range(0, len(kpts), frame_step)]
 
 
def sliding_window_mb(frames, window_size=243, stride=81):
    """Sliding window for MotionBERT: (N, T, 17, 3)"""
    windows = []
    T = len(frames)
    for s in range(0, T - window_size + 1, stride):
        w = np.array(frames[s:s+window_size])
        windows.append(w)
    
    if len(windows) == 0 and T > 0:
        w = np.zeros((window_size, 17, 3))
        w[:T] = np.array(frames)
        windows.append(w)
    return np.array(windows) if windows else np.empty((0, window_size, 17, 3))
 
 
SEQUENCES = {}
for code, name in GENRE_CODES.items():
    sbm = sorted([f for f in all_files if f.startswith(code) and '_sBM_' in f])
    SEQUENCES[name] = {'train': sbm[:16], 'test': sbm[16:20]}
 
X_train, X_test = [], []
y_train, y_test = [], []
 
print('Loading sequences...')
for genre, splits in SEQUENCES.items():
    label = GENRE_TO_IDX[genre]
    for split, X_list, y_list in [('train', X_train, y_train), ('test', X_test, y_test)]:
        for pkl_file in splits[split]:
            path = f'{PKL_DIR}/{pkl_file}'
            if not os.path.exists(path): continue
            frames  = load_sequence(path, frame_step=3)
            windows = sliding_window_mb(frames, WINDOW_SIZE, STRIDE)
            if len(windows) > 0:
                X_list.extend(windows)
                y_list.extend([label] * len(windows))
 
X_train = np.nan_to_num(np.array(X_train))
X_test  = np.nan_to_num(np.array(X_test))
y_train = np.array(y_train)
y_test  = np.array(y_test)
 
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Input format: (N, T={WINDOW_SIZE}, V=17, C=3)')


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
 
try:
    from lib.model.DSTformer import DSTformer
    print('DSTformer imported from MotionBERT')
except ImportError:
    print('Trying alternative import...')
    from model.DSTformer import DSTformer
 
 
class MotionBERTClassifier(nn.Module):
    """
    MotionBERT fine-tuned for dance genre classification.
    Architecture: DSTformer backbone + linear classification head.
    
    Following Zhu et al. (ICCV 2023): 'adapting MotionBERT to different
    downstream tasks with 1-2 linear layers'.
    """
    def __init__(self, num_classes=10, dim_feat=512, pretrained_path=None):
        super().__init__()
        
        self.backbone = DSTformer(
            dim_in=3,
            dim_out=3,
            dim_feat=dim_feat,
            dim_rep=512,
            depth=5,
            num_heads=8,
            mlp_ratio=2,
            num_joints=17,
            maxlen=243,
        )
 
        
        if pretrained_path and os.path.exists(pretrained_path):
            checkpoint = torch.load(pretrained_path, map_location='cpu')
            if 'model_pos' in checkpoint:
                state_dict = checkpoint['model_pos']
            elif 'state_dict' in checkpoint:
                state_dict = checkpoint['state_dict']
            else:
                state_dict = checkpoint
            
            missing, unexpected = self.backbone.load_state_dict(
                {k.replace('module.', ''): v for k, v in state_dict.items()},
                strict=False
            )
            print(f'Pretrained loaded: {len(missing)} missing, {len(unexpected)} unexpected')
        else:
            print('No pretrained weights — training from scratch')
 
        
        self.head = nn.Sequential(
            nn.LayerNorm(dim_feat),
            nn.Linear(dim_feat, num_classes)
        )
 
    def forward(self, x):
        
        
        feat = self.backbone(x, return_rep=True)
        
        feat = feat.mean(dim=[1, 2])
        return self.head(feat)
 
 
class SkeletonDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]
 
 
PRETRAINED = 'checkpoint/pretrain/MB_release.bin'
model = MotionBERTClassifier(
    num_classes=10,
    pretrained_path=PRETRAINED
).to(device)
 
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')


In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score
 
BATCH_SIZE = 16
 
train_loader = DataLoader(SkeletonDataset(X_train, y_train),
                          batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(SkeletonDataset(X_test,  y_test),
                          batch_size=BATCH_SIZE, shuffle=False)
 
print(f'Train batches: {len(train_loader)}')
print(f'Test batches:  {len(test_loader)}')
 
 
def train_phase(model, train_loader, test_loader, num_epochs,
                lr, freeze_backbone=False, label=''):
    if freeze_backbone:
        for param in model.backbone.parameters():
            param.requires_grad = False
        print(f'[{label}] Backbone frozen — training head only')
    else:
        for param in model.backbone.parameters():
            param.requires_grad = True
        print(f'[{label}] Full fine-tuning')
 
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=0.01
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
 
    best_acc   = 0.0
    best_epoch = 0
    history    = []
 
    for epoch in range(num_epochs):
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            out  = model(X_b)
            loss = criterion(out, y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
 
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for X_b, y_b in test_loader:
                out = model(X_b.to(device))
                correct += (out.argmax(1) == y_b.to(device)).sum().item()
                total   += y_b.size(0)
        acc = 100 * correct / total
        history.append(acc)
 
        if acc > best_acc:
            best_acc   = acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), f'best_motionbert_{label}.pt')
 
        if (epoch + 1) % 5 == 0:
            print(f'  Epoch [{epoch+1}/{num_epochs}] Test: {acc:.1f}% | Best: {best_acc:.1f}%')
 
    return best_acc, best_epoch, history
 
 
print('=' * 55)
print('PHASE 1: Head-only fine-tuning')
print('=' * 55)
best1, ep1, hist1 = train_phase(
    model, train_loader, test_loader,
    num_epochs=20, lr=1e-3,
    freeze_backbone=True, label='phase1'
)
print(f'Phase 1 best: {best1:.2f}% at epoch {ep1}')
 
print('\n' + '=' * 55)
print('PHASE 2: Full fine-tuning')
print('=' * 55)
model.load_state_dict(torch.load(f'best_motionbert_phase1.pt'))
best2, ep2, hist2 = train_phase(
    model, train_loader, test_loader,
    num_epochs=30, lr=1e-4,
    freeze_backbone=False, label='phase2'
)
print(f'Phase 2 best: {best2:.2f}% at epoch {ep2}')


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
 
model.load_state_dict(torch.load('best_motionbert_phase2.pt'))
model.eval()
 
all_preds, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        out = model(X_b.to(device))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(y_b.numpy())
 
all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
 
mb_acc = accuracy_score(all_labels, all_preds) * 100
mb_f1  = f1_score(all_labels, all_preds, average='macro') * 100
 
print('=' * 55)
print('MOTIONBERT FINAL RESULTS')
print('=' * 55)
print(f'Accuracy:  {mb_acc:.2f}%')
print(f'Macro F1:  {mb_f1:.2f}%')
print()
print('Classification Report:')
print(classification_report(all_labels, all_preds, target_names=GENRES, digits=4))
 
comparison = pd.DataFrame([
    {'Model': 'SVM',         'Type': 'Traditional ML',  'Acc': 85.96, 'F1': 85.00},
    {'Model': 'RF',          'Type': 'Traditional ML',  'Acc': 85.96, 'F1': 83.84},
    {'Model': 'LSTM',        'Type': 'Deep Learning',   'Acc': 91.23, 'F1': 90.33},
    {'Model': 'ST-GCN',      'Type': 'Deep Learning',   'Acc': 98.68, 'F1': 98.55},
    {'Model': 'MotionBERT',  'Type': 'Pretrained Transformer', 'Acc': round(mb_acc,2), 'F1': round(mb_f1,2)},
])
print('\n=== FULL COMPARISON ===')
print(comparison.to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
 
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
 
axes[0].plot(hist1, color='#2196F3', label='Phase 1 (head only)')
axes[0].plot(range(len(hist1), len(hist1)+len(hist2)),
             hist2, color='#FF9800', label='Phase 2 (full fine-tune)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('MotionBERT Fine-tuning Curves')
axes[0].legend()
axes[0].grid(alpha=0.3)
 
colors = ['#90A4AE','#90A4AE','#2196F3','#4CAF50','#9C27B0']
bars = axes[1].bar(comparison['Model'], comparison['Acc'], color=colors, alpha=0.9)
for bar, v in zip(bars, comparison['Acc']):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.3,
                f'{v:.1f}%', ha='center', fontsize=8, fontweight='bold')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('All Models Comparison')
axes[1].set_ylim(75, 105)
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y', alpha=0.3)
 
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
short_g = [g[:7] for g in GENRES]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Purples',
            xticklabels=short_g, yticklabels=short_g, ax=axes[2])
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('True')
axes[2].set_title('MotionBERT Confusion Matrix')
axes[2].tick_params(axis='x', rotation=25)
 
plt.suptitle(f'MotionBERT Dance Genre Classification — Accuracy: {mb_acc:.1f}%', fontsize=13)
plt.tight_layout()
plt.savefig('motionbert_results.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
import json, shutil
 
comparison.to_csv('full_model_comparison.csv', index=False)
 
summary = {
    'model':       'MotionBERT (DSTformer)',
    'pretrained':  'MB_release.bin (ICCV 2023)',
    'window':      WINDOW_SIZE,
    'fps':         '20fps (frame_step=3)',
    'train':       int(len(X_train)),
    'test':        int(len(X_test)),
    'phase1_best': best1,
    'phase2_best': best2,
    'final_acc':   round(mb_acc, 2),
    'final_f1':    round(mb_f1, 2)
}
with open('motionbert_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
 
files = ['motionbert_results.png', 'full_model_comparison.csv',
         'motionbert_summary.json', 'best_motionbert_phase2.pt']
for f in files:
    if os.path.exists(f):
        shutil.copy(f, f'{AR_DIR}/{f}')
        print(f'Saved: {f}')
 
print(f'\nAll results → AIST_ActionRec_MotionBERT/')
print()
print('=== FINAL SUMMARY ===')
print(comparison.to_string(index=False))
